In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
import d4rl
import argparse
import pickle
import sys
import tqdm
import importlib
import os
from matplotlib import pyplot as plt

# sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from common.normalizer import StandardNormalizer
from common import util
from models.transition_model import TransitionModel
from models.d4rl_world_model import D4RLWorldModel
from common.buffer import ReplayBuffer
from common.functional import dict_batch_generator
from scoring import crps_evaluation

In [5]:
def plot_predictions_rl(next_obs, ground_truth):

    
        input_color = 'tab:blue'
        pred_color = 'tab:orange' #label="input",
        gt_color = 'tab:green'
        rl_color = 'tab:red'

        fig, ax1 = plt.subplots(figsize = (8,5.8), dpi=300)
                                        
        #plot the next_obs all dimensions and ground truth all dimensions 
        for i in range(next_obs.shape[1]):
            ax1.plot(np.arange(len(next_obs)), ground_truth[:, i], color=gt_color, label='Ground Truth')
            ax1.plot(np.arange(len(next_obs)), next_obs[:, i], color=pred_color, label='Prediction' )
        ax1.legend(loc='upper right')
        
        plt.show()

        

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/abiomed/intermediate_data_d4rl/hopper-expert-v0_noisy.pkl"
# data_path = ""
env_name = 'hopper-expert-v0'
env = gym.make(env_name)
if data_path == "":
    dataset = d4rl.qlearning_dataset(env)
else:
    print('loading')
    with open(data_path, 'rb') as f:
        dataset = pickle.load(f)

loading


/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/gym/envs/registration.py:505: UserWarning: WARN: The environment hopper-expert-v0 is out of date. You should consider upgrading to version `v2` with the environment ID `hopper-expert-v2`.
  logger.warn(
/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/d4rl/gym_mujoco/gym_envs.py:13: UserWarning: This environment is deprecated. Please use the most recent version of this environment.
  offline_env.OfflineEnv.__init__(self, **kwargs)
/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/gym/spaces/box.py:78: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [19]:
obs_shape = env.observation_space.shape
action_dim = np.prod(env.action_space.shape)   

offline_buffer = ReplayBuffer(
            buffer_size=len(dataset["observations"]),
            obs_shape=obs_shape,
            obs_dtype=np.float32,
            action_dim=action_dim,
            action_dtype=np.float32
        )

offline_buffer.load_dataset(dataset)

In [20]:
transition_params= {
        "model_batch_size": 256,
        "use_weight_decay": True,
        "optimizer_class": "Adam",
        "learning_rate": 0.001,
        "holdout_ratio": 0.2,
        "inc_var_loss": True,
        "model": {
            "hidden_dims": [200, 200, 200, 200],
            "decay_weights": [0.000025, 0.00005, 0.000075, 0.000075, 0.0001],
            "act_fn": "swish",
            "out_act_fn": "identity",
            "num_elite": 5,
            "ensemble_size": 7
        }
}

In [21]:
mopo_params = {
        "max_epoch": 125,
        "rollout_batch_size": 50000,
        "rollout_mini_batch_size": 10000,
        "model_retain_epochs": 1,
        "num_env_steps_per_epoch": 1000,
        "train_model_interval": 250,
        "max_trajectory_length": 1000,
        "eval_interval": 1000,
        "num_eval_trajectories": 10,
        "snapshot_interval": 2000,
        "model_env_ratio": 0.95,
        "max_model_update_epochs_to_improve": 5,
        "max_model_train_iterations": "None",
        'model_batch_size': 256,
        "rollout_batch_size":50000,
        "rollout_mini_batch_size":1000,
        "model_retain_epochs":1,
        "num_env_steps_per_epoch":1000,
        "max_epoch":100000,
        "max_model_update_epochs_to_improve":5,
        "max_model_train_iterations":np.inf,
        "hold_out_ratio":0.1,
    }

In [22]:
task = env_name.split('-')[0]
import_path = f"static_fns.{task}"
static_fns = importlib.import_module(import_path).StaticFns
transition_model = TransitionModel(
        obs_space=env.observation_space,
        action_space=env.action_space,
        static_fns=static_fns,
        lr=transition_params['learning_rate'],
        device=device,
        **transition_params
    )

transition device cuda


In [23]:
#merge two dictionaries
params = { **transition_params, **mopo_params }

In [8]:
def learn_dynamics(dynamics_model, offline_buffer, params=params):
        # get train and eval data
        model_tot_train_timesteps = 0
        max_sample_size = offline_buffer.get_size
        num_train_data = int(max_sample_size * (1.0 - params["holdout_ratio"]))
        env_data = offline_buffer.sample_all()
        train_data, eval_data = {}, {}
        for key in env_data.keys():
            train_data[key] = env_data[key][:num_train_data]
            eval_data[key] = env_data[key][num_train_data:]
        dynamics_model.reset_normalizers()
        dynamics_model.update_normalizer(train_data['observations'], train_data['actions'])

        # train model
        model_train_iters = 0
        model_train_epochs = 0
        num_epochs_since_prev_best = 0
        break_training = False
        dynamics_model.reset_best_snapshots()

        # init eval_mse_losses
        print("Start training dynamics")
        eval_mse_losses, _ = dynamics_model.eval_data(eval_data, update_elite_models=False)
        print("loss/model_eval_mse_loss", eval_mse_losses.mean(), model_tot_train_timesteps)
        updated = dynamics_model.update_best_snapshots(eval_mse_losses)
       
        while not break_training:
            # starttime = time.time()
            for train_data_batch in dict_batch_generator(train_data, params["model_batch_size"]):
                train_data_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in train_data_batch.items()}
                model_log_infos = dynamics_model.update(train_data_batch)
                model_train_iters += 1
                model_tot_train_timesteps += 1

            eval_mse_losses, _ = dynamics_model.eval_data(eval_data, update_elite_models=False)
            print("loss/model_eval_mse_loss", eval_mse_losses.mean(), model_tot_train_timesteps)
            # print('elapsed time',time.time() - starttime)
            updated = dynamics_model.update_best_snapshots(eval_mse_losses)
            num_epochs_since_prev_best += 1
            if updated:
                model_train_epochs += num_epochs_since_prev_best
                num_epochs_since_prev_best = 0
            if num_epochs_since_prev_best >= params["max_model_update_epochs_to_improve"] or model_train_iters > params["max_model_train_iterations"]\
                    or model_tot_train_timesteps > 800000:
                break
            
        
        #look at load_best_snapshots and model_best_snapshots
        dynamics_model.load_best_snapshots()

        # evaluate data to update the elite models
        dynamics_model.eval_data(eval_data, update_elite_models=True)
        model_log_infos['misc/norm_obs_mean'] = torch.mean(torch.Tensor(dynamics_model.obs_normalizer.mean)).item()
        model_log_infos['misc/norm_obs_var'] = torch.mean(torch.Tensor(dynamics_model.obs_normalizer.var)).item()
        model_log_infos['misc/norm_act_mean'] = torch.mean(torch.Tensor(dynamics_model.act_normalizer.mean)).item()
        model_log_infos['misc/norm_act_var'] = torch.mean(torch.Tensor(dynamics_model.act_normalizer.var)).item()
        model_log_infos['misc/model_train_epochs'] = model_train_epochs
        model_log_infos['misc/model_train_train_steps'] = model_train_iters
        return dynamics_model

In [39]:
#hopper-expert
dynamics_model = learn_dynamics(transition_model, offline_buffer, params)
torch.save(dynamics_model.networks["model"].state_dict(), f'saved_models/{env_name}/transition_model/dynamics_model.pt')

Start training dynamics
loss/model_eval_mse_loss 0.011485869 0
loss/model_eval_mse_loss 0.015805524 3122
loss/model_eval_mse_loss 0.009605297 6244
loss/model_eval_mse_loss 0.009138512 9366
loss/model_eval_mse_loss 0.0084671 12488
loss/model_eval_mse_loss 0.012834102 15610
loss/model_eval_mse_loss 0.0075677163 18732
loss/model_eval_mse_loss 0.0070253867 21854
loss/model_eval_mse_loss 0.007482369 24976
loss/model_eval_mse_loss 0.017063592 28098
loss/model_eval_mse_loss 0.0063525764 31220
loss/model_eval_mse_loss 0.006825262 34342
loss/model_eval_mse_loss 0.006046738 37464
loss/model_eval_mse_loss 0.0058913096 40586
loss/model_eval_mse_loss 0.005943174 43708
loss/model_eval_mse_loss 0.008345503 46830
loss/model_eval_mse_loss 0.0051651336 49952
loss/model_eval_mse_loss 0.004911891 53074
loss/model_eval_mse_loss 0.0070634284 56196
loss/model_eval_mse_loss 0.0058031045 59318
loss/model_eval_mse_loss 0.0047867363 62440
loss/model_eval_mse_loss 0.005958922 65562
loss/model_eval_mse_loss 0.0053

In [9]:
#hopper-expert-noisy
trained_synamics_model = learn_dynamics(transition_model, offline_buffer, params)

Start training dynamics
loss/model_eval_mse_loss 46.296734 0
loss/model_eval_mse_loss 11.13684 3122
loss/model_eval_mse_loss 10.921691 6244
loss/model_eval_mse_loss 11.3531885 9366
loss/model_eval_mse_loss 11.055474 12488
loss/model_eval_mse_loss 11.006547 15610
loss/model_eval_mse_loss 11.191683 18732
loss/model_eval_mse_loss 11.058241 21854


In [ ]:
# torch.save(trained_synamics_model.state_dict(), f'saved_models/{env_name}/dynamics_model.pt')

In [16]:
# env_name = env_name + "_noisy"
# if not os.path.exists(f'saved_models/{env_name}/transition_model'):
#     os.makedirs(f'saved_models/{env_name}/transition_model')
# torch.save(trained_synamics_model.networks["model"].state_dict(), f'saved_models/{env_name}/transition_model/dynamics_model.pt')

In [ ]:
# torch.save(dynamics_model.networks["model"].state_dict(), f'saved_models/{env_name}/dynamics_model.pt')

In [92]:
load_saved_model = transition_model.networks["model"]
env_name2 = env_name +"_noisy"
load_saved_model.load_state_dict(torch.load(f'saved_models/{env_name2}/transition_model/dynamics_model.pt'))
transition_model.model = load_saved_model

/tmp/ipykernel_114411/805166183.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  load_saved_model.load_state_dict(torch.load(f'saved_models/{env_name2}/transition_model/d

In [14]:
load_saved_model = transition_model.networks["model"]

load_saved_model.load_state_dict(torch.load(f'saved_models/{env_name}/transition_model/dynamics_model.pt'))
transition_model.model = load_saved_model

/tmp/ipykernel_114411/3297722881.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  load_saved_model.load_state_dict(torch.load(f'saved_models/{env_name}/transition_model/d

In [24]:
max_sample_size = offline_buffer.get_size
num_train_data = int(max_sample_size * (1.0 - params["holdout_ratio"]))
env_data = offline_buffer.sample_all()
eval_data = {}
for key in env_data.keys():
    eval_data[key] = env_data[key][num_train_data:]



In [15]:
obs = torch.tensor(eval_data['observations'][:2])
act =torch.tensor( eval_data['actions'][:2])
scaled_obs, scaled_act = transition_model.transform_obs_action(obs, act)
model_input = torch.cat([scaled_obs, scaled_act], dim=-1).to(device)
pr = transition_model.model.predict(model_input)

In [25]:
def get_eval(model, D, batch_size = 50, wm=None):
    mse = 0 
    total_batches = len(D['observations']) // batch_size
    for i in (range(total_batches)):
   
        state = D['observations'][i*batch_size:(i+1)*batch_size].copy()
        action = D['actions'][i*batch_size:(i+1)*batch_size].copy()
        next_state = D['next_observations'][i*batch_size:(i+1)*batch_size].copy()

        # Get the next observations and rewards from the transition model
        if wm is None:
            # scaled_obs, scaled_act = transition_model.transform_obs_action(state, action)
            # model_input = torch.cat([scaled_obs, scaled_act], dim=-1).to(device)
            # next_pred = transition_model.model.predict(model_input)
            # print('predicting ensemble model')
            next_pred ,_ ,_ ,_= model.predict(state,action, deterministic = True)
        else:
            # print('predicting world model')
            next_pred = model.predict(state, action)
        mse = nn.MSELoss(reduction='sum')(torch.tensor(next_pred), torch.tensor(next_state))
        
    print("Eval MSE:", mse.item())
    # plot_predictions_rl(next_obs, sample['next_observations'])

In [78]:

get_eval(transition_model, eval_data)

Eval MSE: 0.9784139394760132


In [15]:
get_eval(transition_model, eval_data)

Eval MSE: 15811.59765625


In [14]:
#hopper-expert
env_name = 'hopper-expert-v0'
world_model = D4RLWorldModel(env_name)
world_model.load_model(f'saved_models/hopper-expert-v0/world_model_0.01.pth')
get_eval(world_model, eval_data, wm=True)

/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/gym/envs/registration.py:505: UserWarning: WARN: The environment hopper-expert-v0 is out of date. You should consider upgrading to version `v2` with the environment ID `hopper-expert-v2`.
  logger.warn(
/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/d4rl/gym_mujoco/gym_envs.py:13: UserWarning: This environment is deprecated. Please use the most recent version of this environment.
  offline_env.OfflineEnv.__init__(self, **kwargs)
/home/ubuntu/miniconda3/envs/mopo/lib/python3.8/site-packages/gym/spaces/box.py:78: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
load datafile: 100%|██████████| 5/5 [00:00<00:00, 12.16it/s]


loaded dataset
Using device: cuda:0


/home/ubuntu/mbpo_uq/models/d4rl_world_model.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=f'{self.device}')


Eval MSE: 1.249310851097107


In [26]:
#hopper-expert-noisy
env_name = 'hopper-expert-v0'
world_model = D4RLWorldModel(env_name,)
world_model.load_model(f'saved_models/hopper-expert-v0_noisy/world_model_5.34.pth')
get_eval(world_model, eval_data, wm=True)

load datafile: 100%|██████████| 5/5 [00:01<00:00,  3.46it/s]


loaded dataset
Using device: cuda:0


/home/ubuntu/mbpo_uq/models/d4rl_world_model.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=f'{self.device}')


Eval MSE: 12973.9599609375


# CRPS calculation

In [27]:
def get_crps_list(env_name, data, model, device, batch_size=50, num_samples=50):

    print(f"Starting CRPS list for {env_name}")
    model.model.to(device)
    crps_list = []
    total_batches = len(data['observations']) // batch_size
    # data['observations'] = data['observations'][:51]
    # data['actions'] = data['actions'][:51]
    # data['next_observations'] = data['next_observations'][:51]
    # data['terminals'] = data['terminals'][:51]
    # data['rewards'] = data['rewards'][:51]

    with torch.no_grad():
        for i in tqdm.tqdm(range(total_batches)):
            state = data['observations'][i*batch_size:(i+1)*batch_size]
            action = data['actions'][i*batch_size:(i+1)*batch_size]
            next_state = data['next_observations'][i*batch_size:(i+1)*batch_size]

            states = np.repeat(state, num_samples, axis=0)
            actions = np.repeat(action, num_samples, axis=0)

            pred_input = torch.FloatTensor(np.concatenate([states, actions], axis=1)).to(device)
            
            model_pred = model.model(pred_input).cpu().numpy()
            
            # make it stop at the right length
            for i in range(batch_size):
                crps = crps_evaluation(model_pred[i * num_samples:(i+1) * num_samples], next_state[i])
                crps_list.append(crps.mean())

        # last batch
        if total_batches*batch_size < len(data['observations']):
            state = data['observations'][total_batches*batch_size:]
            action = data['actions'][total_batches*batch_size:]
            next_state = data['next_observations'][total_batches*batch_size:]

            states = np.repeat(state, num_samples, axis=0)
            actions = np.repeat(action, num_samples, axis=0)

            pred_input = torch.FloatTensor(np.concatenate([states, actions], axis=1)).to(device)
            print("pred_input shape:", pred_input.shape)
            model_pred = model.model(pred_input).cpu().numpy()

            for i in range(state.shape[0]):
                crps = crps_evaluation(model_pred[i * num_samples:(i+1) * num_samples], next_state[i])
                crps_list.append(crps.mean())

    crps_list = np.array(crps_list)
    print(crps_list.shape)

    print("MEAN CRPS OVER SAMPLES:", crps_list.mean())
    # verify crps_list is the same length as the dataset
    assert len(crps_list) == len(data['observations']), "crps_list is not the same length as the dataset"
    


In [28]:
#noisy
get_crps_list(env_name, eval_data, world_model, device, num_samples =7)


Starting CRPS list for hopper-expert-v0


100%|██████████| 3996/3996 [00:22<00:00, 178.01it/s]

pred_input shape: torch.Size([49, 14])
(199807,)
MEAN CRPS OVER SAMPLES: 1.4092342


In [17]:
#normal
get_crps_list(env_name, eval_data, world_model, device, num_samples =7)


Starting CRPS list for hopper-expert-v0


100%|██████████| 3996/3996 [00:22<00:00, 179.13it/s]


pred_input shape: torch.Size([49, 14])
(199807,)
MEAN CRPS OVER SAMPLES: 0.033328284


In [52]:
def ensemble_predict_mult(model, obs, act):
    model_input = torch.FloatTensor(np.concatenate([obs, act], axis=1)).to(device)
    pred_diff_means, pred_diff_logvars = model.model.predict(model_input)
    pred_diff_means = pred_diff_means.detach().cpu().numpy()

    ensemble_model_stds = pred_diff_logvars.exp().sqrt().detach().cpu().numpy()
    pred_diff_means = pred_diff_means + np.random.normal(size=pred_diff_means.shape) * ensemble_model_stds    
    pred_diff_samples = pred_diff_means[:, :, :-1]

    next_obs = pred_diff_samples + obs
    return np.swapaxes(next_obs, 0, 1)

In [94]:
def get_crps_list_ensemble(env_name, data, model, device):

    print(f"Starting CRPS list for {env_name}")
    model.model.to(device)
    crps_list = []
    batch_size = 50
    total_batches = len(data['observations']) // batch_size
    num_samples = 7
    
    with torch.no_grad():
        for i in tqdm.tqdm(range(total_batches)):
            state = data['observations'][i*batch_size:(i+1)*batch_size]
            action = data['actions'][i*batch_size:(i+1)*batch_size]
            next_state = data['next_observations'][i*batch_size:(i+1)*batch_size]
            model_pred = ensemble_predict_mult(model, state, action)
                        
            # make it stop at the right length
            for i in range(batch_size):
                crps = crps_evaluation(model_pred[i], next_state[i])
                crps_list.append(crps.mean())

        # last batch
        print(total_batches*batch_size)
        if total_batches*batch_size < len(data['observations']):
            state = data['observations'][total_batches*batch_size:]
            action = data['actions'][total_batches*batch_size:]
            next_state = data['next_observations'][total_batches*batch_size:]

            model_pred = ensemble_predict_mult(model, state, action)
            for i in range(state.shape[0]):
                crps = crps_evaluation(model_pred[i], next_state[i])
                crps_list.append(crps.mean())

    crps_list = np.array(crps_list)
   

    print("MEAN CRPS OVER SAMPLES:", crps_list.mean())
    # verify crps_list is the same length as the dataset
    assert len(crps_list) == len(data['observations']), "crps_list is not the same length as the dataset"
    


In [89]:
#normal
get_crps_list_ensemble(env_name, eval_data, transition_model, device)

Starting CRPS list for hopper-expert-v0


100%|██████████| 3996/3996 [00:28<00:00, 141.39it/s]


199800
tba
tba
tba
tba
tba
tba
tba
MEAN CRPS OVER SAMPLES: 0.007393585512403262


In [95]:
#noisy
get_crps_list_ensemble(env_name, eval_data, transition_model, device)

Starting CRPS list for hopper-expert-v0


100%|██████████| 3996/3996 [00:28<00:00, 140.63it/s]

199800
tba
tba
tba
tba
tba
tba
tba
MEAN CRPS OVER SAMPLES: 1.5734926806994685
